# EDA Fraud

Exploratory analysis for fraud datasets (creditcard + PaySim).

Steps:
- Verify data presence and sizes.
- Inspect label balance and basic stats.
- Sample PaySim for schema and fraud rate.
- Run the training data audit and summarize outputs.



In [ ]:
from __future__ import annotations

import json
import os
import sys
import subprocess
from pathlib import Path

# Resolve repo root from the notebook location.
REPO_ROOT = Path.cwd()
for parent in [REPO_ROOT] + list(REPO_ROOT.parents):
    if (parent / 'scripts').exists() and (parent / 'notebooks').exists():
        REPO_ROOT = parent
        break

# Ensure local modules are importable.
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'src'))

PY = sys.executable

def run(cmd: list[str]) -> None:
    # Run a command from the repo root with PYTHONPATH set.
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(REPO_ROOT / 'src'), str(REPO_ROOT)])
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(REPO_ROOT), check=True, env=env)

def show_json(rel_path: str) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        print(path.read_text(encoding='utf-8', errors='ignore')[:2000])
        return
    print(json.dumps(data, indent=2))

def list_dir(rel_path: str, limit: int = 20) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    print(f'\n{rel_path}/')
    for item in sorted(path.iterdir())[:limit]:
        print(' -', item.name)


In [ ]:
import pandas as pd
from collections import Counter
from pathlib import Path

fraud_root = REPO_ROOT / 'data' / 'raw' / 'fraud'
creditcard_path = fraud_root / 'creditcard.csv'
paysim_dir = fraud_root / 'paysim'

fraud_summary: dict[str, dict[str, object]] = {
    'creditcard': {},
    'paysim': {},
}

def format_bytes(num: int) -> str:
    step = 1024.0
    units = ['B', 'KB', 'MB', 'GB', 'TB']
    size = float(num)
    for unit in units:
        if size < step:
            return f'{size:,.1f} {unit}'
        size /= step
    return f'{size:,.1f} PB'

def list_files(root: Path, *, limit: int = 12) -> None:
    if not root.exists():
        print('Missing:', root)
        return
    files = [p for p in root.rglob('*') if p.is_file()]
    print(f'Files: {len(files)}')
    for path in files[:limit]:
        try:
            size = path.stat().st_size
        except OSError:
            size = 0
        print(' -', path.relative_to(REPO_ROOT), format_bytes(size))

print('Fraud data root:', fraud_root)
list_files(fraud_root)

if creditcard_path.exists():
    fraud_summary['creditcard']['path'] = str(creditcard_path)
    fraud_summary['creditcard']['size_mb'] = round(creditcard_path.stat().st_size / 1024**2, 2)
else:
    print('Missing creditcard.csv:', creditcard_path)

if paysim_dir.exists():
    csvs = sorted(paysim_dir.rglob('*.csv'))
    fraud_summary['paysim']['file_count'] = len(csvs)
    print('PaySim CSVs:', len(csvs))
    for sample in csvs[:5]:
        print(' -', sample.relative_to(REPO_ROOT))
else:
    print('Missing PaySim directory:', paysim_dir)


In [ ]:
# Credit card fraud dataset overview.
if creditcard_path.exists():
    df = pd.read_csv(creditcard_path)
    fraud_summary['creditcard']['rows'] = int(df.shape[0])
    fraud_summary['creditcard']['cols'] = int(df.shape[1])
    print('creditcard.csv shape:', df.shape)
    print('Columns:', list(df.columns))

    missing = df.isna().sum().sort_values(ascending=False)
    print('Missing values (top 10):')
    print(missing.head(10))

    dupes = int(df.duplicated().sum())
    fraud_summary['creditcard']['duplicates'] = dupes
    print('Duplicate rows:', dupes)

    label_col = 'Class' if 'Class' in df.columns else None
    if label_col:
        counts = df[label_col].value_counts().sort_index()
        fraud_ratio = float(counts.get(1, 0) / len(df))
        fraud_summary['creditcard']['label_counts'] = counts.to_dict()
        fraud_summary['creditcard']['fraud_ratio'] = round(fraud_ratio, 6)
        print('Label distribution:')
        print(counts)
        print('Fraud ratio:', fraud_ratio)

        if 'Amount' in df.columns:
            print('Amount stats by label:')
            print(df.groupby(label_col)['Amount'].describe())

        if 'Time' in df.columns:
            print('Time range:', df['Time'].min(), 'to', df['Time'].max())

        corr = df.corr(numeric_only=True)[label_col].sort_values(ascending=False)
        print('Top positive correlations to label:')
        print(corr.head(6))
        print('Top negative correlations to label:')
        print(corr.tail(6))
    else:
        print('No label column found in creditcard.csv')


In [ ]:
# PaySim sample inspection.
if paysim_dir.exists():
    csvs = sorted(paysim_dir.rglob('*.csv'))
    if not csvs:
        print('No PaySim CSV files found.')
    else:
        sample_rows = 50000
        paysim_stats: list[dict[str, object]] = []
        for csv_path in csvs[:3]:
            print(f'\nPreview PaySim: {csv_path.relative_to(REPO_ROOT)}')
            df = pd.read_csv(csv_path, nrows=sample_rows)
            row_count = int(df.shape[0])
            cols = list(df.columns)
            print('Rows (sample):', row_count, '| Columns:', len(cols))
            print('Columns:', cols)
            summary = {
                'file': str(csv_path),
                'rows_sampled': row_count,
                'columns': cols,
            }
            for label in ['isFraud', 'isFlaggedFraud', 'label']:
                if label in df.columns:
                    counts = df[label].value_counts().to_dict()
                    summary[f'{label}_counts'] = counts
                    print(f'{label} counts:', counts)
            paysim_stats.append(summary)
        fraud_summary['paysim']['samples'] = paysim_stats


In [ ]:
# Persist summary for quick reference.
report_dir = REPO_ROOT / 'reports'
report_dir.mkdir(parents=True, exist_ok=True)
summary_path = report_dir / 'eda_fraud_summary.json'
summary_path.write_text(json.dumps(fraud_summary, indent=2))
print('Saved summary to', summary_path)


In [ ]:
# Generate a training data audit
run([PY, 'scripts/training_data_audit.py'])



In [ ]:
# Summarize fraud-related entries from the training data audit.
audit_path = REPO_ROOT / 'reports' / 'TRAINING_DATA.json'
if not audit_path.exists():
    print('Missing:', audit_path)
else:
    audit = json.loads(audit_path.read_text(encoding='utf-8'))
    fraud_items = [
        item for item in audit.get('required', []) + audit.get('optional', [])
        if 'fraud' in str(item.get('name', '')).lower()
    ]
    if not fraud_items:
        print('No fraud entries found in TRAINING_DATA.json')
    else:
        print('Fraud datasets in audit:')
        for item in fraud_items:
            print(' -', item.get('name'), '|', item.get('status'), '|', item.get('path'))


In [ ]:
# Quick artifact index for verification.
for folder in ['models', 'experiments', 'artifacts', 'runs', 'reports', 'logs']:
    path = REPO_ROOT / folder
    if not path.exists():
        continue
    print(f'\n{folder}/')
    for item in sorted(path.iterdir())[:20]:
        print(' -', item.name)
